# Logistic Regression — Dataset Preparation

This notebook reproduces the locked dataset preparation procedure for the Logistic Regression experiment: original dataset verification, conservative cleaning, stratified train/test split, and TF-IDF feature extraction.

**Labels:** `0 = Legitimate`, `1 = Phishing`. Checkpoint assertions identify deviations from the recorded experiment. Summary tables are displayed without publishing raw email messages.


In [ ]:
import os, re
import pandas as pd
import opendatasets as od
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

RANDOM_STATE = 42
TEST_SIZE = 0.20
MIN_CLEAN_LENGTH = 20

EXPECTED_RAW_ROWS = 53668
EXPECTED_CLEAN_ROWS = 53577
EXPECTED_TRAIN_ROWS = 42861
EXPECTED_TEST_ROWS = 10716
EXPECTED_FEATURES = 10000

DATASET_URL = "https://www.kaggle.com/datasets/rohansood98/phishing-email-dataset-nazario-5-and-trec07"
print("Configuration loaded.")


## 1. Download and load

The dataset is the Kaggle **Phishing Email Dataset (Nazario_5 and TREC07)** and the file used is `email_text.csv`.

No Kaggle credentials are stored in this notebook. Authenticate securely when prompted.


In [ ]:
from pathlib import Path

# Colab's default download directory is /content; other environments use the current folder.
base_dir = Path('/content') if Path('/content').is_dir() else Path.cwd()
dataset_dir = base_dir / 'phishing-email-dataset-nazario-5-and-trec07'
dataset_path = dataset_dir / 'email_text.csv'

if not dataset_path.exists():
    od.download(DATASET_URL, data_dir=str(base_dir))

if not dataset_path.exists():
    raise FileNotFoundError(f'Dataset not found at {dataset_path}. Check the Kaggle download location.')

email_text = pd.read_csv(dataset_path)
print('Dataset loaded:', dataset_path)


## 2. Verify the original dataset

The original experiment contained **53,668 emails** before cleaning.


**Table 1. Original dataset class distribution**

| Class | Emails | Percentage |
| --- | ---: | ---: |
| Legitimate (0) | 23,745 | 44.24% |
| Phishing (1) | 29,923 | 55.76% |
| Total | 53,668 | 100.00% |

*Recorded checkpoints; run the cells below to verify against the source dataset.*


In [ ]:
print('DATASET SHAPE:', email_text.shape)
print('DATASET COLUMNS:', email_text.columns.tolist())
print('\nCLASS DISTRIBUTION:')
original_classes = email_text['label'].value_counts().sort_index()
display(pd.DataFrame({
    'Class': ['Legitimate (0)', 'Phishing (1)'],
    'Count': [int(original_classes.get(0, 0)), int(original_classes.get(1, 0))],
    'Percentage': [round(100 * original_classes.get(0, 0) / len(email_text), 2),
                   round(100 * original_classes.get(1, 0) / len(email_text), 2)]
}))
print('\nMISSING VALUES:')
display(email_text.isnull().sum().rename('Missing values').to_frame())
print('\nDUPLICATE EMAIL TEXTS:', email_text['text'].duplicated().sum())

email_text['length'] = email_text['text'].astype(str).str.len()
print('\nEMAIL LENGTH STATISTICS:')
display(email_text['length'].describe().rename('Characters').to_frame())

assert len(email_text) == EXPECTED_RAW_ROWS
assert email_text['label'].isnull().sum() == 0
assert email_text['text'].isnull().sum() == 0
assert email_text['text'].duplicated().sum() == 0
assert int(original_classes.loc[1]) == 29923
assert int(original_classes.loc[0]) == 23745
print('\nOriginal dataset verification passed.')


## 3. Conservative cleaning

The original procedure converts text to string, lowercases it, removes URLs and email addresses, keeps letters/numbers/spaces, collapses whitespace, and removes cleaned records shorter than 20 characters.


**Table 2. Cleaned dataset class distribution**

| Class | Emails | Percentage |
| --- | ---: | ---: |
| Legitimate (0) | 23,739 | 44.31% |
| Phishing (1) | 29,838 | 55.69% |
| Total | 53,577 | 100.00% |

*Recorded checkpoints; run the cells below to verify against the source dataset.*


In [ ]:
email_clean = email_text.copy()
email_clean['text'] = email_clean['text'].astype(str)

def clean_email_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

email_clean['clean_text'] = email_clean['text'].apply(clean_email_text)
email_clean['clean_length'] = email_clean['clean_text'].str.len()
email_clean = email_clean[email_clean['clean_length'] >= MIN_CLEAN_LENGTH].reset_index(drop=True)

print('CLEANED DATASET SHAPE:', email_clean.shape)
clean_classes = email_clean['label'].value_counts().sort_index()
display(pd.DataFrame({
    'Class': ['Legitimate (0)', 'Phishing (1)'],
    'Count': [int(clean_classes.get(0, 0)), int(clean_classes.get(1, 0))],
    'Percentage': [round(100 * clean_classes.get(0, 0) / len(email_clean), 2),
                   round(100 * clean_classes.get(1, 0) / len(email_clean), 2)]
}))
print('\nCLEANED LENGTH STATISTICS:')
display(email_clean['clean_length'].describe().rename('Characters').to_frame())

assert len(email_clean) == EXPECTED_CLEAN_ROWS
assert int(clean_classes.loc[1]) == 29838
assert int(clean_classes.loc[0]) == 23739
assert EXPECTED_RAW_ROWS - len(email_clean) == 91
print('\nCleaning checkpoint passed.')


## 4. Fixed stratified train/test split

The cleaned dataset is divided 80/20 using `random_state=42` and stratification by label. The resulting test set is the reserved official evaluation set.


**Table 3. Stratified training and test partitions**

| Partition | Legitimate (0) | Phishing (1) | Total emails |
| --- | ---: | ---: | ---: |
| Training | 18,991 | 23,870 | 42,861 |
| Testing | 4,748 | 5,968 | 10,716 |
| Total | 23,739 | 29,838 | 53,577 |

*Recorded checkpoints; run the cells below to verify against the source dataset.*


In [ ]:
X = email_clean['clean_text']
y = email_clean['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

split_table = pd.DataFrame({
    'Subset': ['Training', 'Test', 'Total'],
    'Legitimate (0)': [int((y_train == 0).sum()), int((y_test == 0).sum()), int((y == 0).sum())],
    'Phishing (1)': [int((y_train == 1).sum()), int((y_test == 1).sum()), int((y == 1).sum())],
    'Total emails': [len(X_train), len(X_test), len(X)]
})
print('STRATIFIED SPLIT SUMMARY:')
display(split_table)

assert len(X_train) == EXPECTED_TRAIN_ROWS
assert len(X_test) == EXPECTED_TEST_ROWS
assert int((y_train == 1).sum()) == 23870
assert int((y_train == 0).sum()) == 18991
assert int((y_test == 1).sum()) == 5968
assert int((y_test == 0).sum()) == 4748
print('\nSplit checkpoint passed.')


## 5. Locked TF-IDF representation

Configuration: `max_features=10000`, `ngram_range=(1,2)`, `min_df=2`, `max_df=0.95`, `stop_words="english"`.

The vectorizer is fitted **only on training text**. The reserved test text is transformed using the fitted vocabulary.


In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words="english"
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TRAINING TF-IDF SHAPE:", X_train_tfidf.shape)
print("TEST TF-IDF SHAPE:", X_test_tfidf.shape)
print("VOCABULARY SIZE:", len(tfidf_vectorizer.get_feature_names_out()))

assert X_train_tfidf.shape == (EXPECTED_TRAIN_ROWS, EXPECTED_FEATURES)
assert X_test_tfidf.shape == (EXPECTED_TEST_ROWS, EXPECTED_FEATURES)
assert len(tfidf_vectorizer.get_feature_names_out()) == EXPECTED_FEATURES
print("\nTF-IDF checkpoint passed.")


**Table 4. Dataset preparation checkpoints**

| Checkpoint | Recorded value |
| --- | ---: |
| Original emails | 53,668 |
| Cleaned emails | 53,577 |
| Removed during cleaning | 91 |
| Training emails | 42,861 |
| Test emails | 10,716 |
| Training TF-IDF matrix | 42,861 × 10,000 |
| Test TF-IDF matrix | 10,716 × 10,000 |

*Recorded checkpoints; run the cells below to verify against the source dataset.*


## 6. Final checkpoint

The recorded dataset preparation checkpoints are presented in Table 4 above. The code verifies them when run.

**Reproducibility note:** Later notebooks must reuse the fitted TF-IDF vocabulary or recreate it with the exact locked training split and preprocessing procedure.


In [ ]:
print("=" * 70)
print("DATASET PREPARATION COMPLETED SUCCESSFULLY")
print("=" * 70)
print(f"Raw dataset      : {len(email_text):,}")
print(f"Cleaned dataset  : {len(email_clean):,}")
print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")
print(f"Training matrix  : {X_train_tfidf.shape}")
print(f"Testing matrix   : {X_test_tfidf.shape}")
print(f"TF-IDF features  : {len(tfidf_vectorizer.get_feature_names_out()):,}")
